# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print an overview using metadata
md_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'N/A')}\nDescription: {getattr(dataset.metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.  All entities are referenced by their `@id`.

In [ ]:
# List all record sets and their fields (by @id)

record_sets = dataset.metadata.record_sets
print(f"Total record sets found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', '')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    # List fields/columns
    field_ids = [field.id for field in getattr(record_set, 'fields', [])]
    print(f"  Fields (@id): {field_ids}\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's extract data from all record sets into a dict of DataFrames. Use @id as keys.

dataframes = {}
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]  # list of all @id of record sets

for recset_id in record_sets_ids:
    print(f"Loading records for record set {recset_id} ...")
    records = list(dataset.records(record_set=recset_id))
    if len(records) == 0:
        print(f"No records found for {recset_id}.")
        continue
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"Fields: {dataframes[recset_id].columns.tolist()}")
    display(dataframes[recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field references use their `@id` from the overview.

In [ ]:
from pandas.api.types import is_numeric_dtype
import numpy as np

# Automatically detect first record set with data (if no data, skip)
main_record_set_id = None
for rs_id in dataframes:
    if not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id is None:
    raise ValueError('No data found in any record set!')
print(f"Main record set for EDA: {main_record_set_id}")
df = dataframes[main_record_set_id]

# Show available columns and infer numeric fields
print(f"Available columns: {df.columns.tolist()}")

# Try to find a numeric field (@id)
numeric_field_ids = [col for col in df.columns if is_numeric_dtype(df[col]) or np.issubdtype(df[col], np.number)]
if len(numeric_field_ids) == 0:
    # Attempt to coerce float columns if read as strings
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            continue
    numeric_field_ids = [col for col in df.columns if is_numeric_dtype(df[col])]
    
if numeric_field_ids:
    example_numeric_field = numeric_field_ids[0]  # use its full @id
    print(f"Selecting numeric field for analysis: {example_numeric_field}")
    # Threshold is median for demonstration
    threshold = df[example_numeric_field].median()
    filtered_df = df[df[example_numeric_field] > threshold].copy()
    print(f"Filtered records with {example_numeric_field} > {threshold}:")
    display(filtered_df.head())
    filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
    print(f"Normalized {example_numeric_field} for filtered records:")
    display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Pick a group field (string/categorical, different from numeric field)
    group_field = None
    for col in df.columns:
        if col != example_numeric_field and not is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field and group_field in filtered_df:
        grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean().reset_index()
        print(f"Grouped mean of {example_numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[example_numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {example_numeric_field} (by @id)")
    plt.xlabel(example_numeric_field)
    plt.show()
    
    if group_field and group_field in df:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=example_numeric_field, data=df)
        plt.title(f"{example_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library via its Croissant schema. We programmatically listed all record sets and fields using their `@id`, extracted example records, and performed a basic EDA including filtering, normalization, grouping, and visualization on the available numeric data.

- Ensure that you always reference entities (record sets, fields, columns) by their `@id` as required by FAIR and Croissant best practices.
- For more advanced analyses, consider further exploring relationships among record sets and integrating with domain knowledge from the metadata.

**Happy FAIR data science!**